# Parsing
Chromstream presently supports parsing of the folling types of files:
## Chromatograms
- .txt files exported from chromeleon
- .ch files (Agilent)
- .d directories containing multiple .ch files
- .dx files (Agilent)

If your file formats varies,  
## Logfiles
- Logfiles generated by Labview (limited support, preferebly add otehr data as pandas DataFrame)

## Parsing single chromatograms

There are in general two ways of parsing a chromatogram. 1) parsing it directly to a chromatogram object 2) Adding a Chromatogram to a experiment or channel object. You can also construct your own chromatogram object.

For agilent .d directories and .dx files, the simple parsing commands return lists, as these 

### Parsing to an object

the `parse_chromatogram` function can parse individual chromatograms 

In [1]:
# Chromeleon
from pathlib import Path
import chromstream as cs
import pandas as pd
chrompaths = Path("../..") /"tests" / "testdata" / "chroms" 
chromeleon_test = chrompaths / "format_6.txt"
chromatogram_1 = cs.parse_chromatogram(chromeleon_test)

# Agilent
agilent_test_path = chrompaths / "agilent.d" / "FID1A.ch"
chromatogram_2 = cs.parse_chromatogram(agilent_test_path)

# if you know the specific format, you can also use the more specific parser
chromatogram_1 = cs.parse_chromatogram_txt(chromeleon_test) #chromeleon .txt format
chromatogram_2 = cs.parse_agilent_ch(agilent_test_path) # Agilent .ch format

# custom chromatogram
mychromatogram = cs.Chromatogram(
        data=pd.DataFrame({"time": [0, 1, 2], "signal": [10, 20, 15]}), # must be a dataframe with "time" and "signal" columns
        injection_time=pd.Timestamp("2024-01-01 12:00:00"), # pandas Timestamp for injection time
        metadata={"time_unit": "s", "signal_unit": "mV"}, # any dict
        channel="FID",
        path=None
    )

# Agilent .d directory
agilent_d_path = chrompaths / "agilent.d" # returns a list of chromatograms from the .d directory
chromlist_1 = cs.parse_agilent_dot_d(agilent_d_path)

# Agilent .dx file
agilent_dx_path = chrompaths / "test_dx.dx"
chromlist_2 = cs.parse_agilent_dx(agilent_dx_path)

### Adding to an Experiment or Channel
You can also start by inilizing an experiment and using a method to add the chromatogram. 
Note that if you want to add multiple chromatograms at the same time, you need a seperate method. The channel is inferred automatically from the metadata.
When parsing to a channel, the injeciton number needs to be specified

In [ ]:
myexp = cs.Experiment(name="My Experiment")

# From a path
myexp.add_chromatogram(chromeleon_test)

# From a Chromatogram object
myexp.add_chromatogram(chromatogram_1)

# From a dx file or a .d directory (adds all chromatograms in the file/directory)
myexp.add_mult_chromatograms(agilent_d_path)

# adding to channel works analogously
mychannel = cs.ChannelChromatograms(channel="My Channel")
mychannel.add_chromatogram(1, chromatogram_1)